In [ ]:
def costs(preds,y_true,cost_fn):
    cost_fp = 2
    cost_tp = 2
    cost_tn = 0
    return np.sum(preds * y_true * cost_tp * np.ones(y_true.shape[0]) + (1-preds) * y_true * cost_fn + preds * (1-y_true) * cost_fp * np.ones(y_true.shape[0]))

In [ ]:
cost_log = costs(prediction_log,y_test_under,X_test_under.amt)
cost_dt = costs(prediction_dt, y_test_under,X_test_under.amt)
cost_rf = costs(prediction_rf, y_test_under,X_test_under.amt)
cost_bo = costs(prediction_boost, y_test_under,X_test_under.amt)
print(cost_log,cost_dt,cost_rf,cost_bo)

In [ ]:
import joblib
import sys
sys.modules['sklearn.externals.joblib'] = joblib
import six
sys.modules['sklearn.externals.six'] = six

_removed_aliases = {
"bool": bool,
"int": int,
"float": float,
"complex": complex,
"object": object,
"str": str,
}

for name, pytype in _removed_aliases.items():
    if not hasattr(np, name):
        setattr(np, name, pytype)


import inspect
from collections import namedtuple

if not hasattr(inspect, "getargspec"):
    def _getargspec(func):
        fas = inspect.getfullargspec(func) # modern API
        ArgSpec = namedtuple("ArgSpec", "args varargs keywords defaults")
        return ArgSpec(args=fas.args,varargs=fas.varargs,keywords=fas.varkw,
               defaults=fas.defaults,
        ) 
        inspect.getargspec = _getargspec # monkey-patch


from costcla.metrics import cost_loss, savings_score

In [ ]:
fraud_df_sampled = fraud_df.sample(int(len(fraud_df) * 0.2))
cost_fp = 2
cost_fn = fraud_df_sampled['amt']
cost_tp = 2
cost_tn = 0
cost_mat = np.array([cost_fp * np.ones(fraud_df_sampled.shape[0]),
                    cost_fn,
                    cost_tp * np.ones(fraud_df_sampled.shape[0]),
                    cost_tn * np.ones(fraud_df_sampled.shape[0])]).T

In [ ]:
X_train, X_test, y_train, y_test, cost_mat_train, cost_mat_test = train_test_split(fraud_df_sampled.drop('is_fraud', axis=1),fraud_df_sampled.is_fraud, cost_mat,test_size=0.2, random_state=0)

In [ ]:
saving_models = []
saving_models.append(('Logistic Regression',LogisticRegression()))
saving_models.append(('Decision Tree',DecisionTreeClassifier()))
saving_models.append(('Random Forest',RandomForestClassifier()))
saving_models.append(('XGBoost',XGBClassifier()))

In [ ]:
for name, save_model in saving_models:
    sv_model = save_model
    sv_model.fit(X_train, y_train)
    y_pred = sv_model.predict(X_test)
    saving_score_base = savings_score(y_test.values, y_pred, cost_mat_test)
    f1_score_base = f1_score(y_test, y_pred)
    print('{} Saving score: {:.4f} F1 score: {:.4f}'.format(name, saving_score_base,f1_score_base))